# Level2 PPO Training Notebook

这个 notebook 用来单步调试 `lsy_drone_racing/control/train_CleanRL_ppo.py` 里的直接 level2 PPO 训练流程。

推荐顺序：

1. 检查 kernel 和仓库路径。
2. 跑 `debug_rollout` 看 observation 和 reward 分量。
3. 跑很小的 smoke train，确认 PPO 更新不报错。
4. 调大 `num_envs` / `total_timesteps` 正式训练。
5. 用不渲染的本地评估循环快速看 checkpoint 行为。

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print("repo:", ROOT)
print("python:", sys.executable)

repo: /home/miao/repo/lsy_drone_racing
python: /home/miao/repo/lsy_drone_racing/.pixi/envs/gpu/bin/python


如果下面 import 失败，通常是当前 notebook kernel 没选对。`default` 环境没有 `torch`。

正式 GPU 训练请选择：

```text
.pixi/envs/gpu/bin/python
```

CPU debug 或没有 GPU 时可以选择：

```text
.pixi/envs/tests/bin/python
```

或者在终端用 `pixi shell -e tests` 后启动 notebook。


如果 VS Code 仍然不显示 kernel，在项目根目录手动运行其中一个：

```bash
pixi run -e gpu python -m ipykernel install --user --name lsy-drone-racing-gpu --display-name "LSY Drone Racing (gpu)"
pixi run -e tests python -m ipykernel install --user --name lsy-drone-racing-tests --display-name "LSY Drone Racing (tests)"
```

然后重载 VS Code，再选择 `LSY Drone Racing (gpu)` 或 `LSY Drone Racing (tests)`。

In [2]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import torch
import wandb

from lsy_drone_racing.control.train_CleanRL_ppo import (
    Agent,
    Args,
    debug_rollout,
    make_envs,
    train_ppo,
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("wandb:", wandb.__version__)

W0520 19:47:15.093939  225534 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0520 19:47:15.097829  225173 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


torch: 2.8.0+cu128
cuda available: True
wandb: 0.26.0


## 统一配置

In [3]:
CONFIG = "level2.toml"
MODEL_PATH = ROOT / "lsy_drone_racing/control/ppo_level2_notebook_gatepass1.ckpt"

JAX_DEVICE = "gpu"
CUDA = True

WANDB_ENABLED = True
WANDB_PROJECT_NAME = "ADR-PPO-Racing"
WANDB_ENTITY = None  # 如果你们用 team，在这里填 team/entity 名称
WANDB_MODE = "online"  # 没网时可改成 "offline"

NUM_ENVS_DEBUG = 4
NUM_ENVS_TRAIN = 256
NUM_STEPS = 32
TOTAL_TIMESTEPS_TRAIN = 20_000_000
LEARNING_RATE = 3e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
UPDATE_EPOCHS = 5
NUM_MINIBATCHES = 8
ENT_COEF = 0.02
TARGET_KL = 0.03

REWARD_KWARGS = dict(
    progress_coef=10.0,
    gate_axis_coef=12.0,
    near_gate_coef=0.0,
    gate_bonus=30.0,
    finish_bonus=80.0,
    missed_gate_penalty=8.0,
    crash_penalty=50.0,
    obstacle_coef=1.5,
    obstacle_margin=0.35,
    time_penalty=0.05,
    act_coef=0.005,
    d_act_th_coef=0.02,
    d_act_xy_coef=0.05,
    rpy_coef=0.03,
)

def build_args(**overrides):
    base = dict(
        config=CONFIG,
        cuda=CUDA,
        jax_device=JAX_DEVICE,
        wandb_project_name=WANDB_PROJECT_NAME,
        wandb_entity=WANDB_ENTITY,
        num_steps=NUM_STEPS,
        gamma=GAMMA,
        gae_lambda=GAE_LAMBDA,
        update_epochs=UPDATE_EPOCHS,
        num_minibatches=NUM_MINIBATCHES,
        n_obs=2,
        **REWARD_KWARGS,
    )
    base.update(overrides)
    return Args.create(**base)

def torch_device(args):
    return torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")

MODEL_PATH

PosixPath('/home/miao/repo/lsy_drone_racing/lsy_drone_racing/control/ppo_level2_notebook_gatepass1.ckpt')

## W&B 登录

第一次运行会要求登录。不要把 API key 写进 notebook；按提示在输入框里粘贴即可。

如果你只想本地记录，把上面配置里的 `WANDB_MODE` 改成 `"offline"`。

In [4]:
os.environ["WANDB_MODE"] = WANDB_MODE
os.environ["WANDB_NOTEBOOK_NAME"] = str(ROOT / "notebooks/train_level2_ppo.ipynb")

if WANDB_ENABLED:
    if WANDB_MODE != "offline":
        wandb.login()
    print(f"W&B enabled: project={WANDB_PROJECT_NAME}, entity={WANDB_ENTITY}")
else:
    print("W&B disabled")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/miao/.netrc.
wandb: Currently logged in as: xiaochenmiao (xiaochenmiao-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B enabled: project=ADR-PPO-Racing, entity=None


## 1. 检查 observation / reward

这一格不会训练，只用零动作跑几步。重点看：

- `obs_dim=91` 是否稳定。
- reward 里 `progress / gate_bonus / crash / obstacle / smooth` 是否合理。
- 是否有非有限值或一开始大量 crash。

In [5]:
debug_args = build_args(
    num_envs=NUM_ENVS_DEBUG,
    total_timesteps=NUM_ENVS_DEBUG * NUM_STEPS,
)

debug_rollout(
    debug_args,
    n_steps=20,
    device=torch_device(debug_args),
    jax_device=debug_args.jax_device,
)

/home/miao/repo/lsy_drone_racing/.pixi/envs/gpu/lib/python3.13/site-packages/jax/_src/abstract_arrays.py:135: RuntimeWarning: overflow encountered in cast
  return literals.TypedNdArray(np.asarray(x, dtype), weak_type=False)


[obs-debug] obs_dim=91 num_envs=4
[obs-debug] pos_z                      slice=000:001 mean= 0.017 std= 0.007 min= 0.011 max= 0.028
[obs-debug] vel_body                   slice=001:004 mean= 0.000 std= 0.000 min= 0.000 max= 0.000
[obs-debug] ang_vel                    slice=004:007 mean= 0.000 std= 0.000 min= 0.000 max= 0.000
[obs-debug] rotmat                     slice=007:016 mean= 0.333 std= 0.472 min=-0.079 max= 1.000
[obs-debug] target_progress            slice=016:017 mean= 0.000 std= 0.000 min= 0.000 max= 0.000
[obs-debug] gate_current_corners_body  slice=017:029 mean= 0.720 std= 1.050 min=-0.795 max= 2.281
[obs-debug] gate_next_corners_body     slice=029:041 mean= 1.235 std= 1.077 min=-0.346 max= 2.839
[obs-debug] obstacles_body             slice=041:053 mean= 0.685 std= 1.189 min=-1.680 max= 2.670
[obs-debug] gates_visited              slice=053:057 mean= 0.000 std= 0.000 min= 0.000 max= 0.000
[obs-debug] obstacles_visited          slice=057:061 mean= 0.000 std= 0.000 min= 0.0

## 2. 手动创建环境看 shape

如果你改了 `RaceObservation`，先跑这个确认 shape 和 action space。

In [6]:
shape_args = build_args(num_envs=NUM_ENVS_DEBUG)
envs = make_envs(
    config=shape_args.config,
    num_envs=shape_args.num_envs,
    jax_device=shape_args.jax_device,
    torch_device=torch_device(shape_args),
    coefs=REWARD_KWARGS | {"n_obs": shape_args.n_obs},
    debug_obs=True,
    debug_reward_every=0,
)
obs, info = envs.reset(seed=shape_args.seed)
print("obs shape:", tuple(obs.shape))
print("single obs space:", envs.single_observation_space)
print("single action space:", envs.single_action_space)
envs.close()

[obs-debug] obs_dim=91 num_envs=4
[obs-debug] pos_z                      slice=000:001 mean= 0.017 std= 0.007 min= 0.011 max= 0.028
[obs-debug] vel_body                   slice=001:004 mean= 0.000 std= 0.000 min= 0.000 max= 0.000
[obs-debug] ang_vel                    slice=004:007 mean= 0.000 std= 0.000 min= 0.000 max= 0.000
[obs-debug] rotmat                     slice=007:016 mean= 0.333 std= 0.472 min=-0.079 max= 1.000
[obs-debug] target_progress            slice=016:017 mean= 0.000 std= 0.000 min= 0.000 max= 0.000
[obs-debug] gate_current_corners_body  slice=017:029 mean= 0.720 std= 1.050 min=-0.795 max= 2.281
[obs-debug] gate_next_corners_body     slice=029:041 mean= 1.235 std= 1.077 min=-0.346 max= 2.839
[obs-debug] obstacles_body             slice=041:053 mean= 0.685 std= 1.189 min=-1.680 max= 2.670
[obs-debug] gates_visited              slice=053:057 mean= 0.000 std= 0.000 min= 0.000 max= 0.000
[obs-debug] obstacles_visited          slice=057:061 mean= 0.000 std= 0.000 min= 0.0

## 3. PPO smoke train

这个只训练几十步，目标是确认 rollout、GAE、反传、保存都不报错。

In [7]:
smoke_args = build_args(
    num_envs=2,
    num_steps=8,
    total_timesteps=64,
    jax_device="cpu",
    cuda=False,
)

train_ppo(
    smoke_args,
    model_path=Path("/tmp/ppo_level2_notebook_smoke.ckpt"),
    device=torch_device(smoke_args),
    jax_device=smoke_args.jax_device,
    wandb_enabled=False,
)

Training on device: cpu | Environment device: cpu
Iter 1/4 took 4.22 seconds
Iter 2/4 took 0.30 seconds
Iter 3/4 took 0.36 seconds
Iter 4/4 took 0.36 seconds
Training for 64 steps took 10.95 seconds.
model saved to /tmp/ppo_level2_notebook_smoke.ckpt


[]

## 4. 正式训练

下一轮训练先用 `2_000_000` steps 看趋势。RTX4060 笔记本建议从 256 env 开始；如果显存还有余量，再试 512 env。

如果 W&B 里 reward 没有任何趋势，先回头调 obs/reward，不要直接加到 10M。

In [8]:
train_args = build_args(
    num_envs=NUM_ENVS_TRAIN,
    num_steps=NUM_STEPS,
    total_timesteps=TOTAL_TIMESTEPS_TRAIN,
    learning_rate=LEARNING_RATE,
    ent_coef=ENT_COEF,
    target_kl=TARGET_KL,
)

if WANDB_ENABLED and wandb.run is not None:
    wandb.finish()

train_ppo(
    train_args,
    model_path=MODEL_PATH,
    device=torch_device(train_args),
    jax_device=train_args.jax_device,
    wandb_enabled=WANDB_ENABLED,
)

print("saved:", MODEL_PATH)
if WANDB_ENABLED and wandb.run is not None:
    print("wandb:", wandb.run.url)
    wandb.finish()

Training on device: cuda | Environment device: gpu


/home/miao/repo/lsy_drone_racing/.pixi/envs/gpu/lib/python3.13/site-packages/jax/_src/abstract_arrays.py:135: RuntimeWarning: overflow encountered in cast
  return literals.TypedNdArray(np.asarray(x, dtype), weak_type=False)


Iter 1/2441 took 19.54 seconds
Iter 2/2441 took 1.41 seconds
Iter 3/2441 took 1.47 seconds
Iter 4/2441 took 1.61 seconds
Iter 5/2441 took 1.57 seconds
Iter 6/2441 took 1.33 seconds
Iter 7/2441 took 1.36 seconds
Iter 8/2441 took 1.38 seconds
Iter 9/2441 took 1.61 seconds
Iter 10/2441 took 1.45 seconds
Iter 11/2441 took 1.51 seconds
Iter 12/2441 took 1.77 seconds
Iter 13/2441 took 1.63 seconds
Iter 14/2441 took 1.54 seconds
Iter 15/2441 took 1.95 seconds
Iter 16/2441 took 1.91 seconds
Iter 17/2441 took 1.78 seconds
Iter 18/2441 took 1.80 seconds
Iter 19/2441 took 1.53 seconds
Iter 20/2441 took 1.57 seconds
Iter 21/2441 took 1.61 seconds
Iter 22/2441 took 1.76 seconds
Iter 23/2441 took 1.37 seconds
Iter 24/2441 took 4.19 seconds
Iter 25/2441 took 6.00 seconds
Iter 26/2441 took 4.86 seconds
Iter 27/2441 took 5.67 seconds
Iter 28/2441 took 5.52 seconds
Iter 29/2441 took 5.20 seconds
Iter 30/2441 took 3.71 seconds
Iter 31/2441 took 3.73 seconds
Iter 32/2441 took 4.27 seconds
Iter 33/2441 too

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Iter 2441/2441 took 0.96 seconds
Training for 19996672 steps took 2645.93 seconds.
model saved to /home/miao/repo/lsy_drone_racing/lsy_drone_racing/control/ppo_level2_notebook_gatepass1.ckpt
saved: /home/miao/repo/lsy_drone_racing/lsy_drone_racing/control/ppo_level2_notebook_gatepass1.ckpt
wandb: https://wandb.ai/xiaochenmiao-technical-university-of-munich/ADR-PPO-Racing/runs/gre2ukhn


charts/SPS,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▅▆▆▇▆▆▇▇███
charts/learning_rate,███▇▇▇▆▆▆▆▅▅▅▅▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▂▂▂▂▂▂▁▁▁
global_step,▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇█
losses/approx_kl,█▅▆▆▆▇▄▅▆▆▅█▆█▇▇▆▇▅▇▄▅▄▆▆▄▆▄▃▄▇▂▅▂▃▂▂▁▁▁
losses/clipfrac,▅▇▅█▆▆▇▅▅▆▅▄▇▆▆▆▅▅▄▃▃▄▅▄▃▄▂▃▄▃▂▁▂▁▁▁▁▁▁▁
losses/entropy,▁▂▂▃▃▄▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇███████████▇▇▇▇▇
losses/explained_variance,▁▁▃▄▂▂▅▃▂▅▄▄▃▅▅▄▃▄▄▆▅▅▆▅▅▅▆▇▆▆▆▇▆▆▆▇▆█▇▆
losses/old_approx_kl,▄▄▃▆▅▄▃▄▄█▃▅▅▆▇▄▇▃▃▃▃▁▄▆▂▃▅▂▃▄▃▃▂▁▂▁▁▁▁▁
losses/policy_loss,▅▆▄▇▅▂▆▄▄█▅▆▃▅▃▇▆▄▁▄▄▅▆▇▃▆█▆▆▆▇▆▇▂▅▆▇▅▆▇
losses/value_loss,█▃▂▃▃▂▂▂▂▃▂▂▁▂▃▁▃▃▂▄▂▃▃▃▃▃▂▃▂▂▂▁▂▂▂▂▂▂▂▂
+25,...


## 5. 快速评估 checkpoint

这里评估的是训练 wrapper 下的 reward/length，不等于 `scripts/evaluate.py` 的比赛完成时间。真正比赛评估还需要把 checkpoint 接到 controller 里。

这个评估循环不调用 render，适合 notebook/headless 环境。

In [9]:
eval_args = build_args(num_envs=1, jax_device="cpu", cuda=False)
device = torch_device(eval_args)
eval_env = make_envs(
    config=eval_args.config,
    num_envs=1,
    jax_device=eval_args.jax_device,
    torch_device=device,
    coefs=REWARD_KWARGS | {"n_obs": eval_args.n_obs},
)
agent = Agent(eval_env.single_observation_space.shape, eval_env.single_action_space.shape).to(device)
agent.load_state_dict(torch.load(MODEL_PATH, map_location=device))
agent.eval()

episode_rewards = []
episode_lengths = []
with torch.no_grad():
    for episode in range(3):
        obs, _ = eval_env.reset(seed=eval_args.seed + episode + 1)
        done = torch.zeros(1, dtype=torch.bool, device=device)
        ep_reward = 0.0
        steps = 0
        while not done.any() and steps < 1500:
            action, _, _, _ = agent.get_action_and_value(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = eval_env.step(action)
            done = terminated | truncated
            ep_reward += reward[0].item()
            steps += 1
        episode_rewards.append(ep_reward)
        episode_lengths.append(steps)
        print(f"episode={episode + 1} reward={ep_reward:.2f} length={steps}")
eval_env.close()

print("mean reward:", float(np.mean(episode_rewards)))
print("mean length:", float(np.mean(episode_lengths)))

/home/miao/repo/lsy_drone_racing/.pixi/envs/gpu/lib/python3.13/site-packages/jax/_src/abstract_arrays.py:135: RuntimeWarning: overflow encountered in cast
  return literals.TypedNdArray(np.asarray(x, dtype), weak_type=False)


episode=1 reward=-35.42 length=1500
episode=2 reward=-38.52 length=1500
episode=3 reward=-31.88 length=1500
mean reward: -35.27358335008224
mean length: 1500.0


## 调参入口

优先调 `REWARD_KWARGS`：

- 如果不往 gate 走：提高 `progress_coef`。
- 如果只靠近不穿门：提高 `gate_bonus`，降低 `near_gate_coef`。
- 如果动作抖：提高 `d_act_xy_coef` / `d_act_th_coef`。
- 如果太保守：降低 `obstacle_coef` / `crash_penalty`。
- 如果学会悬停拖时间：提高 `time_penalty`。